# Seller cheating analysis

This notebook scans one or more `transactions.jsonl` outputs and summarizes, for each seller ID you choose:

- how many times that seller cheated as the seller (`outcome_ok == 0`)
- the average price of those cheating transactions

Edit `OUTPUT_ROOTS` and `TARGET_SELLER_IDS` in the config cell, then run the notebook.

In [5]:
import json
from pathlib import Path
import pandas as pd

In [6]:
def find_transaction_files(output_roots):
    files = []
    seen = set()

    for root in output_roots:
        root_path = Path(root)
        if not root_path.exists():
            print(f'Skipping missing path: {root_path}')
            continue

        for tx_file in root_path.rglob('transactions.jsonl'):
            if tx_file.is_file():
                resolved = tx_file.resolve()
                if resolved not in seen:
                    seen.add(resolved)
                    files.append(tx_file)

    return sorted(files)


def summarize_transactions(transaction_files, target_seller_ids):
    target_ids = {int(value) for value in target_seller_ids}

    if not target_ids:
        raise ValueError('TARGET_SELLER_IDS is empty. Add one or more seller IDs in the config cell.')

    per_file_rows = []
    overall = {
        seller_id: {
            'seller_tx_count': 0,
            'cheat_count': 0,
            'cheat_price_sum': 0.0,
        }
        for seller_id in sorted(target_ids)
    }
    
    num_files = len(transaction_files)

    for tx_file in transaction_files:
        local = {
            seller_id: {
                'seller_tx_count': 0,
                'cheat_count': 0,
                'cheat_price_sum': 0.0,
            }
            for seller_id in sorted(target_ids)
        }
        transactions_scanned = 0

        with tx_file.open('r', encoding='utf-8') as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue

                try:
                    tx = json.loads(line)
                except json.JSONDecodeError:
                    continue

                transactions_scanned += 1

                try:
                    seller_id = int(tx.get('seller'))
                except (TypeError, ValueError):
                    continue

                if seller_id not in target_ids:
                    continue
                local[seller_id]['seller_tx_count'] += 1

                if int(tx.get('outcome_ok', 1)) != 0:
                    continue

                try:
                    price = float(tx.get('price'))
                except (TypeError, ValueError):
                    continue

                local[seller_id]['cheat_count'] += 1
                local[seller_id]['cheat_price_sum'] += price

        for seller_id in sorted(target_ids):
            bucket = local[seller_id]
            cheat_count = bucket['cheat_count']
            avg_price = (bucket['cheat_price_sum'] / cheat_count) if cheat_count else None

            per_file_rows.append({
                'output_folder': str(tx_file.parent),
                'transaction_file': str(tx_file),
                'seller_id': seller_id,
                'seller_tx_count': bucket['seller_tx_count'],
                'cheat_count': cheat_count,
                'avg_cheat_price': avg_price,
                'transactions_scanned': transactions_scanned,
            })

            overall_bucket = overall[seller_id]
            overall_bucket['seller_tx_count'] += bucket['seller_tx_count']
            overall_bucket['cheat_count'] += cheat_count
            overall_bucket['cheat_price_sum'] += bucket['cheat_price_sum']

    overall_rows = []
    for seller_id in sorted(target_ids):
        bucket = overall[seller_id]
        cheat_count = bucket['cheat_count']
        overall_rows.append({
            'seller_id': seller_id,
            'avg_seller_tx_count': bucket['seller_tx_count'] / num_files if num_files else None,
            'avg_cheat_count': cheat_count / num_files if num_files else None,
            'avg_cheat_price': (bucket['cheat_price_sum'] / cheat_count) if cheat_count else None,
        })

    return per_file_rows, overall_rows


def as_frame(rows):
    if pd is None:
        return rows
    return pd.DataFrame(rows)

In [7]:
def run(OUTPUT_ROOTS, TARGET_SELLER_IDS):
    transaction_files = find_transaction_files(OUTPUT_ROOTS)
    print(f'Found {len(transaction_files)} transaction log(s).')

    if not transaction_files:
        raise FileNotFoundError('No transactions.jsonl files found under the configured output roots.')

    per_file_rows, overall_rows = summarize_transactions(transaction_files, TARGET_SELLER_IDS)

    # per_file_df = as_frame(per_file_rows).sort_values(['seller_id', 'output_folder'])
    overall_df = as_frame(overall_rows).sort_values(['seller_id'])
    # print('Per-output summary:')
    # display(per_file_df)
    # print('Overall summary across all selected outputs:')
    # display(overall_df)
    print('\nOverall summary statistics:')
    print("Tx count: ", format(overall_df['avg_seller_tx_count'].mean(), ".2f"), "+-",format(overall_df['avg_seller_tx_count'].std(), ".2f"))
    print("Cheat count: ", format(overall_df['avg_cheat_count'].mean(), ".2f"), "+-",format(overall_df['avg_cheat_count'].std(), ".2f"))
    print("Cheat price: ", format(overall_df['avg_cheat_price'].mean(), ".2f"), "+-",format(overall_df['avg_cheat_price'].std(), ".2f"))


### Periodic

In [8]:
# Edit these settings before running.
# You can point this at one folder, several folders, or leave it as [Path("out")] to scan everything under out/.
# We'll analyze the same seller IDs across D1, D2 and D3 for the same algorithms.
ALGORITHMS = ['eigen', 'mean', 'shape']
VARIANTS = ['D1', 'D2', 'D3']
# Put one or more seller IDs here. Example: [3, 7, 12]
TARGET_SELLER_IDS = [270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284]

for d in VARIANTS:
    for algo in ALGORITHMS:
        OUTPUT_ROOTS = [Path(f"out\\D\\{d}\\{algo}")]

        print("===================================")
        print(f'\n=== Analyzing {algo} on {d}, periodic ===')
        run(OUTPUT_ROOTS, TARGET_SELLER_IDS)
        print("===================================")



=== Analyzing eigen on D1, periodic ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  175.19 +- 4.38
Cheat count:  29.13 +- 1.17
Cheat price:  1.20 +- 0.05

=== Analyzing mean on D1, periodic ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  159.68 +- 4.47
Cheat count:  26.69 +- 1.35
Cheat price:  1.22 +- 0.06

=== Analyzing shape on D1, periodic ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  190.51 +- 7.17
Cheat count:  31.87 +- 1.80
Cheat price:  1.20 +- 0.04

=== Analyzing eigen on D2, periodic ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  174.49 +- 3.37
Cheat count:  39.91 +- 2.01
Cheat price:  1.21 +- 0.03

=== Analyzing mean on D2, periodic ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  149.57 +- 5.98
Cheat count:  33.91 +- 1.36
Cheat price:  1.21 +- 0.04

=== Analyzing shape on D2, periodic ===
Found 10 transaction log(s).

Overall summary statistics:
Tx 

### Price-based

In [9]:
ALGORITHMS = ['eigen', 'mean', 'shape']
VARIANTS = ['D1', 'D2', 'D3']

TARGET_SELLER_IDS = [285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299]

for d in VARIANTS:
    for algo in ALGORITHMS:
        OUTPUT_ROOTS = [Path(f"out\\D\\{d}\\{algo}")]

        print("===================================")
        print(f'\n=== Analyzing {algo} on {d}, price-based ===')
        run(OUTPUT_ROOTS, TARGET_SELLER_IDS)
        print("===================================")



=== Analyzing eigen on D1, price-based ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  175.00 +- 2.89
Cheat count:  39.88 +- 1.23
Cheat price:  2.08 +- 0.05

=== Analyzing mean on D1, price-based ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  145.13 +- 5.17
Cheat count:  33.45 +- 1.50
Cheat price:  2.08 +- 0.06

=== Analyzing shape on D1, price-based ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  189.06 +- 4.55
Cheat count:  43.73 +- 2.35
Cheat price:  2.13 +- 0.06

=== Analyzing eigen on D2, price-based ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  173.30 +- 4.56
Cheat count:  39.82 +- 1.73
Cheat price:  2.08 +- 0.04

=== Analyzing mean on D2, price-based ===
Found 10 transaction log(s).

Overall summary statistics:
Tx count:  146.48 +- 4.94
Cheat count:  33.26 +- 1.72
Cheat price:  2.11 +- 0.06

=== Analyzing shape on D2, price-based ===
Found 10 transaction log(s).

Overall summa